<a href="https://colab.research.google.com/github/liuxiaohu0511/lance-demo/blob/develop/lance_data_evolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pylance


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 MB 19.3 MB/s eta 0:00:00


In [2]:
import shutil
import lance
import numpy as np
import pandas as pd
import pyarrow as pa
import duckdb

Lance 支持传统的架构演变：在数据集中添加、删除和更改列。这些操作中的大多数都可以在不重写数据集中数据文件的情况下执行，这样是非常高效的。此外，Lance 支持数据演进 ，可以使用新列数据回填现有的数据行，而无需重写数据集中的数据文件，适合 在ML 特征工程等场景。

通常，模式更改将与大多数其他并发写入操作发生冲突。例如，如果您在其他人向数据集追加数据时更改了数据集的架构，则您的架构更改或追加将失败，具体取决于作的顺序。因此，建议在没有发生其他写入时执行架构更改。


**添加新的列**



In [3]:
# 在现有数据集中添加新列
# 在生产环境中，我们常见的一个用例是在数据集中添加一个新列而不填充它。这有助于稍后运行一个大型分布式作业来懒加载地填充该列。
# 为此，您可以使用 lance.LanceDataset.add_columns 方法使用 pyarrow.Field 或 pyarrow.Schema 添加列

# 这里只更新数据集的元数据，所以操作非常快

table = pa.table({"id": pa.array([1, 2, 3])})
dataset = lance.write_dataset(table, "null_columns")

# With pyarrow Field
dataset.add_columns(pa.field("embedding", pa.list_(pa.float32(), 128)))
assert dataset.schema == pa.schema([
    ("id", pa.int64()),
    ("embedding", pa.list_(pa.float32(), 128)),
])

# With pyarrow Schema
dataset.add_columns(pa.schema([
    ("label", pa.string()),
    ("score", pa.float32()),
]))
assert dataset.schema == pa.schema([
    ("id", pa.int64()),
    ("embedding", pa.list_(pa.float32(), 128)),
    ("label", pa.string()),
    ("score", pa.float32()),
])

In [4]:
# 上面做了三次修改schema的操作，版本数是3
dataset.versions()

[{'version': 1,
  'timestamp': datetime.datetime(2025, 10, 23, 1, 49, 44, 408696),
  'metadata': {'total_data_file_rows': '3',
   'total_data_files': '1',
   'total_deletion_file_rows': '0',
   'total_deletion_files': '0',
   'total_files_size': '282',
   'total_fragments': '1',
   'total_rows': '3'}},
 {'version': 2,
  'timestamp': datetime.datetime(2025, 10, 23, 1, 49, 44, 411247),
  'metadata': {'total_data_file_rows': '3',
   'total_data_files': '1',
   'total_deletion_file_rows': '0',
   'total_deletion_files': '0',
   'total_files_size': '282',
   'total_fragments': '1',
   'total_rows': '3'}},
 {'version': 3,
  'timestamp': datetime.datetime(2025, 10, 23, 1, 49, 44, 412574),
  'metadata': {'total_data_file_rows': '3',
   'total_data_files': '1',
   'total_deletion_file_rows': '0',
   'total_deletion_files': '0',
   'total_files_size': '282',
   'total_fragments': '1',
   'total_rows': '3'}}]

**回填数据**

新列可以在单个操作中添加并填充，使用 lance.LanceDataset.add_columns 方法。有两种方式指定如何填充新列：首先，为每个新列提供一个 SQL 表达式；其次，提供一个生成新列数据的函数。


In [5]:
table = pa.table({"name": pa.array(["Alice", "Bob", "Carla"])})
dataset = lance.write_dataset(table, "names")
dataset.add_columns({
    "hash": "sha256(name)",
    "status": "'active'",
})
print(dataset.to_table().to_pandas())
#     name                                               hash  status
# 0  Alice  b';\xc5\x10b\x97<E\x8dZo-\x8dd\xa0#$cT\xad~\x0...  active
# 1    Bob  b'\xcd\x9f\xb1\xe1H\xcc\xd8D.Z\xa7I\x04\xccs\x...  active
# 2  Carla  b'\xad\x8d\x83\xff\xd8+Z\x8e\xd4)\xe8Y+\\\xb3\...  active

    name                                               hash  status
0  Alice  b';\xc5\x10b\x97<E\x8dZo-\x8dd\xa0#$cT\xad~\x0...  active
1    Bob  b'\xcd\x9f\xb1\xe1H\xcc\xd8D.Z\xa7I\x04\xccs\x...  active
2  Carla  b'\xad\x8d\x83\xff\xd8+Z\x8e\xd4)\xe8Y+\\\xb3\...  active


您还可以提供一个 Python 函数来生成新列数据。例如，这可以用于计算新的嵌入列。此函数应接受一个 PyArrow RecordBatch 并返回一个 PyArrow RecordBatch 或 Pandas DataFrame。该函数将为数据集中的每个批次调用一次。

如果函数计算成本高昂且可能失败，建议在 UDF 中设置一个检查点文件。此检查点文件在每次调用后保存 UDF 的状态，以便如果 UDF 失败，可以从最后一个检查点重新启动。请注意，此文件可能相当大，因为它需要存储整个数据文件中的未保存结果。

In [10]:
import lance
import pyarrow as pa
import numpy as np
import pandas as pd

# 1. 创建原始数据集
table = pa.table({"id": pa.array([1, 2, 3])})
dataset = lance.write_dataset(table, "ids", mode="overwrite")

# 2. 定义UDF：将二维向量转为Arrow的list类型（一维嵌套结构）
@lance.batch_udf(checkpoint_file="embedding_checkpoint.sqlite")
def add_random_vector(batch):
    # 生成二维向量 (num_rows, 128)
    embeddings = np.random.rand(batch.num_rows, 128).astype("float32")
    # 转换为嵌套列表（每个元素是128维向量的list），使其成为一维数组（元素是list）
    embedding_list = embeddings.tolist()
    # 返回DataFrame，其中"embedding"列是一维的list数组
    return pd.DataFrame({"embedding": embedding_list})

# 3. 添加新列（此时类型正确）
dataset.add_columns(add_random_vector)  # 注意：add_columns返回新数据集，需重新赋值

# 验证结果
print("新增列的类型：", dataset.schema.field("embedding").type)
print("数据示例：", dataset.to_table()["embedding"].to_pylist())

新增列的类型： list<item: double>
数据示例： [[0.9065735340118408, 0.4709378778934479, 0.410791277885437, 0.14615993201732635, 0.4239366054534912, 0.1325686275959015, 0.8342584371566772, 0.8787670135498047, 0.17840208113193512, 0.8593159317970276, 0.10999795794487, 0.6523574590682983, 0.9472725987434387, 0.004088298883289099, 0.20879533886909485, 0.44146403670310974, 0.14451223611831665, 0.03568897768855095, 0.6173192262649536, 0.3242412805557251, 0.058462828397750854, 0.9034062027931213, 0.5619414448738098, 0.16831329464912415, 0.921381950378418, 0.2786414921283722, 0.4814731478691101, 0.23857490718364716, 0.12396157532930374, 0.037131454795598984, 0.8690242171287537, 0.250392347574234, 0.20237727463245392, 0.3420955240726471, 0.730695366859436, 0.6075003743171692, 0.503409206867218, 0.9119893312454224, 0.6323542594909668, 0.786242663860321, 0.06404051929712296, 0.017449531704187393, 0.2332059144973755, 0.4133065342903137, 0.4796012043952942, 0.12827321887016296, 0.1355706751346588, 0.25985360145

如果您已经预计算了一个或多个新列，可以使用 lance.LanceDataset.merge 方法将它们添加到现有数据集中。这允许在不重写整个数据集的情况下填充额外的列。

要使用 merge 方法，提供一个包含您想要添加的列的新数据集，以及用于将新数据与现有数据集连接的列名。

In [11]:
# 例如，假设我们有一个包含嵌入和 id 的数据集：
table = pa.table({
   "id": pa.array([1, 2, 3]),
   "embedding": pa.array([np.array([1, 2, 3]), np.array([4, 5, 6]),
                          np.array([7, 8, 9])])
})
dataset = lance.write_dataset(table, "embeddings", mode="overwrite")

现在如果我们想添加我们生成的标签列，可以通过合并一个新的表格来实现：

In [12]:
new_data = pa.table({
   "id": pa.array([1, 2, 3]),
   "label": pa.array(["horse", "rabbit", "cat"])
})
dataset.merge(new_data, "id")
print(dataset.to_table().to_pandas())
#    id  embedding   label
# 0   1  [1, 2, 3]   horse
# 1   2  [4, 5, 6]  rabbit
# 2   3  [7, 8, 9]     cat

   id  embedding   label
0   1  [1, 2, 3]   horse
1   2  [4, 5, 6]  rabbit
2   3  [7, 8, 9]     cat


**刪除列**

您可以使用 lance.LanceDataset.drop_columns 方法从数据集中删除列。**这是一个仅涉及元数据的操作，不会删除磁盘上的数据**。这使得操作非常快速。

In [14]:
table = pa.table({"id": pa.array([1, 2, 3]),
                 "name": pa.array(["Alice", "Bob", "Carla"]),
                  "age": pa.array([20,23,30])})
dataset = lance.write_dataset(table, "names", mode="overwrite")
dataset.drop_columns(["name"])
print(dataset.schema)
dataset.to_table().to_pandas()
# id: int64

id: int64
age: int64


,id,age
0,1,20
1,2,23
2,3,30


为了实际上从磁盘删除数据，必须重写文件以删除列，然后删除旧文件。
这可以**通过使用 lance.dataset.DatasetOptimizer.compact_files然后跟 lance.LanceDataset.cleanup_old_versions**来完成。

**重命名列**

列可以使用 lance.LanceDataset.alter_columns 方法重命名。

In [16]:
table = pa.table({"id": pa.array([1, 2, 3])})
dataset = lance.write_dataset(table, "ids2")
dataset.alter_columns({"path": "id", "name": "new_id"})
print(dataset.to_table().to_pandas())
#    new_id
# 0       1
# 1       2
# 2       3

   new_id
0       1
1       2
2       3


这同样适用于嵌套列。要指定嵌套列，使用点（ . ）来分隔嵌套的层级。例如：

In [20]:
data = [
  {"meta": {"id": 1, "name": "Alice"}},
  {"meta": {"id": 2, "name": "Bob"}},
]
schema = pa.schema([
    ("meta", pa.struct([
        ("id", pa.int32()),
        ("name", pa.string()),
    ]))
])
dataset = lance.write_dataset(data, "nested_rename", mode='overwrite')
dataset.alter_columns({"path": "meta.id", "name": "new_id"})
print(dataset.to_table().to_pandas())
print(dataset.schema)
#                                  meta
# 0  {'new_id': 1, 'name': 'Alice'}
# 1    {'new_id': 2, 'name': 'Bob'}

                             meta
0  {'new_id': 1, 'name': 'Alice'}
1    {'new_id': 2, 'name': 'Bob'}
meta: struct<new_id: int64, name: string>
  child 0, new_id: int64
  child 1, name: string


**强制类型转换**

除了更改列名外，您还可以使用 lance.LanceDataset.alter_columns 方法更改列的数据类型。这需要将那个列重写到新的数据文件中，但不需要重写其他列。

此方法可用于更改列的向量类型。例如，我们可以将 float32 嵌入列更改为 float16 列以节省磁盘空间，但会以降低精度为代价：

In [22]:
table = pa.table({
   "id": pa.array([1, 2, 3]),
   "embedding": pa.FixedShapeTensorArray.from_numpy_ndarray(
       np.random.rand(3, 128).astype("float32"))
})
dataset = lance.write_dataset(table, "embeddings2", mode='overwrite')
dataset.alter_columns({"path": "embedding",
                       "data_type": pa.list_(pa.float16(), 128)})
print(dataset.schema)
print(dataset.to_table().to_pandas())

# id: int64
# embedding: fixed_size_list<item: halffloat>[128]
#   child 0, item: halffloat

id: int64
embedding: fixed_size_list<item: halffloat>[128]
  child 0, item: halffloat
   id                                          embedding
0   1  [0.8154, 0.607, 0.734, 0.5073, 0.2236, 0.6865,...
1   2  [0.5127, 0.3289, 0.33, 0.8887, 0.655, 0.155, 0...
2   3  [0.4956, 0.8984, 0.0627, 0.7734, 0.2954, 0.841...
